# 07 — Retention Prioritization

## Soft E-commerce Customer Segmentation and Retention Prioritization

This notebook uses the final selected Fuzzy C-Means (FCM) model results to:

1. Validate customer membership information.
2. Calculate the Segment Ambiguity Score.
3. Integrate Customer Lifetime Value (CLV) and Churn Risk.
4. Develop a transparent retention prioritization method.
5. Identify high-priority customers for retention actions.
6. Generate final customer-level outputs for business use.

### Selected Final Clustering Model
- Algorithm: Fuzzy C-Means (FCM)
- Number of clusters (K): 4
- Fuzziness parameter (m): 1.10
- Initialization: K-Means-informed

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Project paths
# ---------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", PROCESSED_DIR)
print("Results directory:", RESULTS_DIR)

print("\nDirectories exist:")
print("Processed:", PROCESSED_DIR.exists())
print("Results:", RESULTS_DIR.exists())
print("Models:", MODELS_DIR.exists())

Project root: d:\FDM\soft-ecommerce-customer-segmentation
Processed data directory: d:\FDM\soft-ecommerce-customer-segmentation\data\processed
Results directory: d:\FDM\soft-ecommerce-customer-segmentation\results

Directories exist:
Processed: True
Results: True
Models: True


## 1. Load Final FCM Membership Results

The selected FCM model provides a membership degree for each customer across the four clusters. These membership values will be used to calculate customer segment ambiguity.

In [2]:
# ---------------------------------------------------------
# Load final FCM membership results
# ---------------------------------------------------------

dev_membership_path = RESULTS_DIR / "fcm_development_memberships_with_uncertainty.csv"
holdout_membership_path = RESULTS_DIR / "fcm_holdout_memberships.csv"

print("Development membership file exists:", dev_membership_path.exists())
print("Holdout membership file exists:", holdout_membership_path.exists())

fcm_dev = pd.read_csv(dev_membership_path)
fcm_holdout = pd.read_csv(holdout_membership_path)

print("\nDevelopment membership shape:", fcm_dev.shape)
print("Holdout membership shape:", fcm_holdout.shape)

print("\nDevelopment columns:")
print(fcm_dev.columns.tolist())

print("\nHoldout columns:")
print(fcm_holdout.columns.tolist())

display(fcm_dev.head())
display(fcm_holdout.head())

Development membership file exists: True
Holdout membership file exists: True

Development membership shape: (40000, 10)
Holdout membership shape: (10000, 7)

Development columns:
['customer_id', 'cluster_1_membership', 'cluster_2_membership', 'cluster_3_membership', 'cluster_4_membership', 'assigned_cluster', 'max_membership', 'cluster_name', 'membership_margin', 'normalized_entropy']

Holdout columns:
['customer_id', 'cluster_1_membership', 'cluster_2_membership', 'cluster_3_membership', 'cluster_4_membership', 'assigned_cluster', 'max_membership']


,customer_id,cluster_1_membership,cluster_2_membership,cluster_3_membership,cluster_4_membership,assigned_cluster,max_membership,cluster_name,membership_margin,normalized_entropy
0,CUST-000001,0.011244,0.040699,0.017737,0.930320,4,0.930320,Low-Engagement Customers,0.889621,0.230452
1,CUST-000002,0.979630,0.014919,0.003786,0.001665,1,0.979630,Inactive / High-Churn-Risk Customers,0.964711,0.082709
2,CUST-000003,0.002256,0.043667,0.950386,0.003691,3,0.950386,High-Value Active Customers,0.906718,0.158348
3,CUST-000004,0.000132,0.003518,0.990960,0.005389,3,0.990960,High-Value Active Customers,0.985571,0.041989
4,CUST-000006,0.000801,0.810517,0.172988,0.015694,2,0.810517,Low-Purchase High-Conversion Customers,0.637529,0.392919


,customer_id,cluster_1_membership,cluster_2_membership,cluster_3_membership,cluster_4_membership,assigned_cluster,max_membership
0,CUST-000005,0.000849,0.774176,0.015216,0.209758,2,0.774176
1,CUST-000007,0.000136,0.006404,0.006538,0.986922,4,0.986922
2,CUST-000008,0.000069,0.979889,0.018418,0.001624,2,0.979889
3,CUST-000039,0.000232,0.037172,0.002988,0.959608,4,0.959608
4,CUST-000041,0.001731,0.065024,0.875331,0.057914,3,0.875331


## 2. Validate FCM Membership Results

Before calculating segment ambiguity, the FCM membership outputs are validated to ensure that:

- every customer has four membership values,
- membership values are between 0 and 1,
- memberships sum approximately to 1 for each customer,
- customer IDs are unique,
- development and holdout customers do not overlap.

In [3]:
membership_cols = [
    "cluster_1_membership",
    "cluster_2_membership",
    "cluster_3_membership",
    "cluster_4_membership"
]

# ---------------------------------------------------------
# Basic validation
# ---------------------------------------------------------

print("DEVELOPMENT")
print("Rows:", len(fcm_dev))
print("Unique customer IDs:", fcm_dev["customer_id"].nunique())
print("Missing membership values:",
      fcm_dev[membership_cols].isna().sum().sum())

print("Membership minimum:",
      fcm_dev[membership_cols].min().min())
print("Membership maximum:",
      fcm_dev[membership_cols].max().max())

dev_membership_sum = fcm_dev[membership_cols].sum(axis=1)

print("Membership sum range:",
      dev_membership_sum.min(),
      "to",
      dev_membership_sum.max())

print("\nHOLDOUT")
print("Rows:", len(fcm_holdout))
print("Unique customer IDs:", fcm_holdout["customer_id"].nunique())
print("Missing membership values:",
      fcm_holdout[membership_cols].isna().sum().sum())

print("Membership minimum:",
      fcm_holdout[membership_cols].min().min())
print("Membership maximum:",
      fcm_holdout[membership_cols].max().max())

holdout_membership_sum = fcm_holdout[membership_cols].sum(axis=1)

print("Membership sum range:",
      holdout_membership_sum.min(),
      "to",
      holdout_membership_sum.max())

# ---------------------------------------------------------
# Check development/holdout overlap
# ---------------------------------------------------------

overlap = set(fcm_dev["customer_id"]) & set(fcm_holdout["customer_id"])

print("\nDevelopment/Holdout customer overlap:", len(overlap))

# ---------------------------------------------------------
# Assertions
# ---------------------------------------------------------

assert fcm_dev["customer_id"].is_unique
assert fcm_holdout["customer_id"].is_unique

assert fcm_dev[membership_cols].isna().sum().sum() == 0
assert fcm_holdout[membership_cols].isna().sum().sum() == 0

assert ((fcm_dev[membership_cols] >= 0) &
        (fcm_dev[membership_cols] <= 1)).all().all()

assert ((fcm_holdout[membership_cols] >= 0) &
        (fcm_holdout[membership_cols] <= 1)).all().all()

assert np.allclose(dev_membership_sum, 1.0, atol=1e-5)
assert np.allclose(holdout_membership_sum, 1.0, atol=1e-5)

assert len(overlap) == 0

print("\n✓ All FCM membership validation checks passed.")

DEVELOPMENT
Rows: 40000
Unique customer IDs: 40000
Missing membership values: 0
Membership minimum: 1.0098124624555821e-07
Membership maximum: 0.999995056761546
Membership sum range: 0.9999999999999993 to 1.0000000000000004

HOLDOUT
Rows: 10000
Unique customer IDs: 10000
Missing membership values: 0
Membership minimum: 1.8711227773618899e-07
Membership maximum: 0.9999789611230274
Membership sum range: 0.9999999999999994 to 1.0000000000000004

Development/Holdout customer overlap: 0

✓ All FCM membership validation checks passed.


## 3. Segment Ambiguity Score

FCM assigns each customer a membership degree for every cluster. A customer with one dominant membership has a relatively clear segment assignment, while a customer with similar memberships across several clusters is more ambiguous.

Three uncertainty measures are considered:

- **Maximum Membership** — strength of the customer's strongest cluster membership.
- **Membership Margin** — difference between the highest and second-highest memberships.
- **Normalized Entropy** — overall uncertainty across all four cluster memberships.

Normalized entropy is used as the final **Segment Ambiguity Score** because it considers the complete membership distribution.

The score ranges approximately from:

- **0 = very clear segment membership**
- **1 = highly ambiguous segment membership**

In [4]:
# ---------------------------------------------------------
# Calculate FCM uncertainty measures
# ---------------------------------------------------------

def add_uncertainty_measures(df, membership_cols):
    result = df.copy()

    memberships = result[membership_cols].to_numpy(dtype=float)

    # Highest membership
    sorted_memberships = np.sort(memberships, axis=1)
    result["max_membership"] = sorted_memberships[:, -1]

    # Difference between highest and second-highest memberships
    result["membership_margin"] = (
        sorted_memberships[:, -1] - sorted_memberships[:, -2]
    )

    # Normalized entropy
    safe_memberships = np.clip(memberships, 1e-12, 1.0)

    result["normalized_entropy"] = (
        -np.sum(
            safe_memberships * np.log(safe_memberships),
            axis=1
        )
        / np.log(len(membership_cols))
    )

    # Final Segment Ambiguity Score
    result["segment_ambiguity_score"] = result["normalized_entropy"]

    return result


fcm_dev_final = add_uncertainty_measures(
    fcm_dev,
    membership_cols
)

fcm_holdout_final = add_uncertainty_measures(
    fcm_holdout,
    membership_cols
)

print("Development ambiguity range:")
print(
    fcm_dev_final["segment_ambiguity_score"].min(),
    "to",
    fcm_dev_final["segment_ambiguity_score"].max()
)

print("\nHoldout ambiguity range:")
print(
    fcm_holdout_final["segment_ambiguity_score"].min(),
    "to",
    fcm_holdout_final["segment_ambiguity_score"].max()
)

print("\nDevelopment ambiguity summary:")
display(
    fcm_dev_final["segment_ambiguity_score"]
    .describe()
    .to_frame()
)

print("Holdout ambiguity summary:")
display(
    fcm_holdout_final["segment_ambiguity_score"]
    .describe()
    .to_frame()
)

Development ambiguity range:
5.0850330075715115e-05 to 0.9954633665831383

Holdout ambiguity range:
0.00019149578669687566 to 0.9959692130078237

Development ambiguity summary:


,segment_ambiguity_score
count,40000.000000
mean,0.348725
std,0.224738
min,0.000051
25%,0.147745
50%,0.337565
75%,0.531161
max,0.995463


Holdout ambiguity summary:


,segment_ambiguity_score
count,10000.000000
mean,0.350578
std,0.223853
min,0.000191
25%,0.151482
50%,0.343523
75%,0.532785
max,0.995969


## 4. Validate the Segment Ambiguity Score

The Segment Ambiguity Score should increase when customer cluster membership becomes less decisive.

Therefore, ambiguity is expected to have a negative relationship with:

- **Maximum membership** — ambiguous customers should have a weaker dominant cluster.
- **Membership margin** — ambiguous customers should have a smaller difference between their first and second strongest cluster memberships.

This validation checks whether the entropy-based ambiguity score behaves consistently with these expectations.

In [5]:
# ---------------------------------------------------------
# Validate Segment Ambiguity Score
# ---------------------------------------------------------

validation_cols = [
    "segment_ambiguity_score",
    "max_membership",
    "membership_margin"
]

print("Development correlations:")
dev_corr = fcm_dev_final[validation_cols].corr()
display(dev_corr.round(4))

print("\nHoldout correlations:")
holdout_corr = fcm_holdout_final[validation_cols].corr()
display(holdout_corr.round(4))

Development correlations:


,segment_ambiguity_score,max_membership,membership_margin
segment_ambiguity_score,1.0000,-0.9531,-0.9265
max_membership,-0.9531,1.0000,0.9910
membership_margin,-0.9265,0.9910,1.0000



Holdout correlations:


,segment_ambiguity_score,max_membership,membership_margin
segment_ambiguity_score,1.0000,-0.9518,-0.9243
max_membership,-0.9518,1.0000,0.9909
membership_margin,-0.9243,0.9909,1.0000


In [8]:
from IPython.display import display as ipy_display

# ---------------------------------------------------------
# Inspect most ambiguous and clearest development customers
# ---------------------------------------------------------

display_cols = [
    "customer_id",
    *membership_cols,
    "assigned_cluster",
    "max_membership",
    "membership_margin",
    "segment_ambiguity_score"
]

most_ambiguous = (
    fcm_dev_final
    .sort_values("segment_ambiguity_score", ascending=False)
    [display_cols]
    .head(10)
)

clearest = (
    fcm_dev_final
    .sort_values("segment_ambiguity_score", ascending=True)
    [display_cols]
    .head(10)
)

print("10 MOST AMBIGUOUS CUSTOMERS")
ipy_display(most_ambiguous)

print("\n10 CLEAREST CUSTOMERS")
ipy_display(clearest)

10 MOST AMBIGUOUS CUSTOMERS


,customer_id,cluster_1_membership,cluster_2_membership,cluster_3_membership,cluster_4_membership,assigned_cluster,max_membership,membership_margin,segment_ambiguity_score
18088,CUST-022667,0.262444,0.241732,0.209974,0.285851,4,0.285851,0.023407,0.995463
12890,CUST-016138,0.257652,0.235390,0.292507,0.214451,3,0.292507,0.034854,0.995216
6676,CUST-008398,0.226551,0.296528,0.262309,0.214613,2,0.296528,0.034219,0.994121
24222,CUST-030338,0.234187,0.317084,0.233320,0.215409,2,0.317084,0.082897,0.991430
6598,CUST-008300,0.291391,0.282203,0.181046,0.245359,1,0.291391,0.009188,0.988593
4831,CUST-006067,0.304563,0.262670,0.276655,0.156113,1,0.304563,0.027908,0.980070
546,CUST-000678,0.291460,0.320133,0.223861,0.164546,2,0.320133,0.028673,0.978117
22071,CUST-027608,0.173116,0.263269,0.220279,0.343336,4,0.343336,0.080067,0.977611
9801,CUST-012261,0.204248,0.196064,0.368859,0.230829,3,0.368859,0.138030,0.973944
39646,CUST-049571,0.186779,0.185721,0.358131,0.269369,3,0.358131,0.088762,0.971741



10 CLEAREST CUSTOMERS


,customer_id,cluster_1_membership,cluster_2_membership,cluster_3_membership,cluster_4_membership,assigned_cluster,max_membership,membership_margin,segment_ambiguity_score
31170,CUST-039046,0.999995,0.000001,2.430454e-06,0.000001,1,0.999995,0.999993,0.000051
15546,CUST-019486,0.999994,0.000003,3.059562e-07,0.000002,1,0.999994,0.999991,0.000057
8424,CUST-010570,0.999992,0.000002,1.220399e-06,0.000005,1,0.999992,0.999987,0.000083
36959,CUST-046225,0.999990,0.000004,2.286775e-06,0.000003,1,0.999990,0.999986,0.000095
8698,CUST-010913,0.999987,0.000007,2.110692e-06,0.000004,1,0.999987,0.999980,0.000122
27819,CUST-034864,0.999978,0.000004,4.077561e-06,0.000014,1,0.999978,0.999964,0.000201
5518,CUST-006952,0.999976,0.000016,3.205348e-06,0.000004,1,0.999976,0.999960,0.000216
23806,CUST-029815,0.999972,0.000005,1.915852e-05,0.000003,1,0.999972,0.999953,0.000246
22243,CUST-027830,0.999969,0.000015,8.780385e-06,0.000007,1,0.999969,0.999954,0.000275
3241,CUST-004076,0.999967,0.000003,5.646523e-06,0.000024,1,0.999967,0.999943,0.000285


## 5. Integrate Business Variables

Customer Lifetime Value (CLV) and Churn Risk were intentionally excluded from the clustering model to prevent them from directly determining the behavioral customer segments.

After segmentation, these variables are reintroduced as business-level information for retention prioritization.

The next stage examines:

- **Customer Lifetime Value (CLV)** — the economic value associated with the customer.
- **Churn Risk Score** — the customer's estimated risk of leaving.

These variables will later be combined with Segment Ambiguity to support retention prioritization.

In [9]:
# ---------------------------------------------------------
# Load business data
# ---------------------------------------------------------

dev_business_path = PROCESSED_DIR / "development_business_data.csv"
holdout_business_path = PROCESSED_DIR / "holdout_business_data.csv"

dev_business = pd.read_csv(dev_business_path)
holdout_business = pd.read_csv(holdout_business_path)

print("Development business data:", dev_business.shape)
print("Holdout business data:", holdout_business.shape)

print("\nDevelopment columns:")
print(dev_business.columns.tolist())

print("\nHoldout columns:")
print(holdout_business.columns.tolist())

print("\nDevelopment missing values:")
print(dev_business.isna().sum())

print("\nHoldout missing values:")
print(holdout_business.isna().sum())

ipy_display(dev_business.head())

Development business data: (40000, 3)
Holdout business data: (10000, 3)

Development columns:
['customer_id', 'customer_lifetime_value_usd', 'churn_risk_score']

Holdout columns:
['customer_id', 'customer_lifetime_value_usd', 'churn_risk_score']

Development missing values:
customer_id                    0
customer_lifetime_value_usd    0
churn_risk_score               0
dtype: int64

Holdout missing values:
customer_id                    0
customer_lifetime_value_usd    0
churn_risk_score               0
dtype: int64


,customer_id,customer_lifetime_value_usd,churn_risk_score
0,CUST-000001,21281.07,39
1,CUST-000002,28347.21,42
2,CUST-000003,103909.84,0
3,CUST-000004,171318.41,14
4,CUST-000006,52302.46,5


## 6. Merge Segmentation and Business Information

The FCM membership results are joined with CLV and churn-risk information using `customer_id`.

The merge is validated to ensure that every segmented customer has exactly one corresponding business record and that no customers are lost during the integration.

In [10]:
# ---------------------------------------------------------
# Validate customer alignment before merging
# ---------------------------------------------------------

print("Development IDs match:",
      set(fcm_dev_final["customer_id"]) ==
      set(dev_business["customer_id"]))

print("Holdout IDs match:",
      set(fcm_holdout_final["customer_id"]) ==
      set(holdout_business["customer_id"]))

assert dev_business["customer_id"].is_unique
assert holdout_business["customer_id"].is_unique

assert set(fcm_dev_final["customer_id"]) == set(dev_business["customer_id"])
assert set(fcm_holdout_final["customer_id"]) == set(holdout_business["customer_id"])

# ---------------------------------------------------------
# Merge
# ---------------------------------------------------------

dev_retention = fcm_dev_final.merge(
    dev_business,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

holdout_retention = fcm_holdout_final.merge(
    holdout_business,
    on="customer_id",
    how="left",
    validate="one_to_one"
)

print("\nDevelopment merged shape:", dev_retention.shape)
print("Holdout merged shape:", holdout_retention.shape)

print("\nMissing CLV after merge:",
      dev_retention["customer_lifetime_value_usd"].isna().sum(),
      holdout_retention["customer_lifetime_value_usd"].isna().sum())

print("Missing churn risk after merge:",
      dev_retention["churn_risk_score"].isna().sum(),
      holdout_retention["churn_risk_score"].isna().sum())

print("\n✓ Segmentation and business data merged successfully.")

Development IDs match: True
Holdout IDs match: True

Development merged shape: (40000, 13)
Holdout merged shape: (10000, 12)

Missing CLV after merge: 0 0
Missing churn risk after merge: 0 0

✓ Segmentation and business data merged successfully.


In [11]:
# ---------------------------------------------------------
# Examine business-variable distributions
# ---------------------------------------------------------

business_cols = [
    "customer_lifetime_value_usd",
    "churn_risk_score",
    "segment_ambiguity_score"
]

print("DEVELOPMENT SUMMARY")
ipy_display(
    dev_retention[business_cols]
    .describe()
    .round(3)
)

print("\nHOLDOUT SUMMARY")
ipy_display(
    holdout_retention[business_cols]
    .describe()
    .round(3)
)

print("\nDEVELOPMENT SKEWNESS")
ipy_display(
    dev_retention[business_cols]
    .skew()
    .round(3)
    .to_frame("skewness")
)

print("\nCORRELATIONS")
ipy_display(
    dev_retention[business_cols]
    .corr()
    .round(3)
)

DEVELOPMENT SUMMARY


,customer_lifetime_value_usd,churn_risk_score,segment_ambiguity_score
count,40000.000,40000.000,40000.000
mean,57779.062,26.789,0.349
std,52431.869,19.188,0.225
min,0.000,0.000,0.000
25%,15459.958,10.000,0.148
50%,42253.030,25.000,0.338
75%,87274.468,40.000,0.531
max,287564.230,100.000,0.995



HOLDOUT SUMMARY


,customer_lifetime_value_usd,churn_risk_score,segment_ambiguity_score
count,10000.000,10000.000,10000.000
mean,57494.097,27.050,0.351
std,52460.938,19.283,0.224
min,0.000,0.000,0.000
25%,15585.458,10.000,0.151
50%,42186.395,25.000,0.344
75%,85169.850,40.000,0.533
max,294634.150,100.000,0.996



DEVELOPMENT SKEWNESS


,skewness
customer_lifetime_value_usd,1.145
churn_risk_score,0.502
segment_ambiguity_score,0.221



CORRELATIONS


,customer_lifetime_value_usd,churn_risk_score,segment_ambiguity_score
customer_lifetime_value_usd,1.000,-0.039,0.051
churn_risk_score,-0.039,1.000,-0.005
segment_ambiguity_score,0.051,-0.005,1.000


## 7. Normalize Retention Priority Components

The three retention-priority signals use different numerical scales. Therefore, they are converted to comparable values between 0 and 1 before being combined.

- **CLV component:** transformed using its percentile position in the development population. This reduces the influence of the right-skewed CLV distribution while preserving customer value ranking.
- **Churn component:** divided by 100 because the original churn-risk score is already defined on a 0–100 scale.
- **Ambiguity component:** used directly because normalized entropy already ranges approximately from 0 to 1.

The development CLV distribution is used as the reference when transforming holdout customers so that information from the holdout set is not used to define the transformation.

In [12]:
# ---------------------------------------------------------
# Normalize retention-priority components
# ---------------------------------------------------------

# Development CLV percentile
dev_retention["clv_component"] = (
    dev_retention["customer_lifetime_value_usd"]
    .rank(method="average", pct=True)
)

# Use DEVELOPMENT CLV distribution as the reference
# for holdout customers
dev_clv_sorted = np.sort(
    dev_retention["customer_lifetime_value_usd"].to_numpy()
)

holdout_clv = holdout_retention[
    "customer_lifetime_value_usd"
].to_numpy()

holdout_retention["clv_component"] = (
    np.searchsorted(
        dev_clv_sorted,
        holdout_clv,
        side="right"
    ) / len(dev_clv_sorted)
)

# Churn risk: 0–100 -> 0–1
dev_retention["churn_component"] = (
    dev_retention["churn_risk_score"] / 100.0
)

holdout_retention["churn_component"] = (
    holdout_retention["churn_risk_score"] / 100.0
)

# Ambiguity is already normalized
dev_retention["ambiguity_component"] = (
    dev_retention["segment_ambiguity_score"]
)

holdout_retention["ambiguity_component"] = (
    holdout_retention["segment_ambiguity_score"]
)

component_cols = [
    "clv_component",
    "churn_component",
    "ambiguity_component"
]

print("DEVELOPMENT COMPONENT SUMMARY")
ipy_display(
    dev_retention[component_cols]
    .describe()
    .round(3)
)

print("\nHOLDOUT COMPONENT SUMMARY")
ipy_display(
    holdout_retention[component_cols]
    .describe()
    .round(3)
)

# Validation
assert (
    dev_retention[component_cols]
    .apply(lambda x: x.between(0, 1))
    .all()
    .all()
)

assert (
    holdout_retention[component_cols]
    .apply(lambda x: x.between(0, 1))
    .all()
    .all()
)

print("\n✓ All retention components are within the 0–1 range.")

DEVELOPMENT COMPONENT SUMMARY


,clv_component,churn_component,ambiguity_component
count,40000.000,40000.000,40000.000
mean,0.500,0.268,0.349
std,0.289,0.192,0.225
min,0.003,0.000,0.000
25%,0.250,0.100,0.148
50%,0.500,0.250,0.338
75%,0.750,0.400,0.531
max,1.000,1.000,0.995



HOLDOUT COMPONENT SUMMARY


,clv_component,churn_component,ambiguity_component
count,10000.000,10000.000,10000.000
mean,0.499,0.270,0.351
std,0.286,0.193,0.224
min,0.005,0.000,0.000
25%,0.252,0.100,0.151
50%,0.500,0.250,0.344
75%,0.742,0.400,0.533
max,1.000,1.000,0.996



✓ All retention components are within the 0–1 range.


## 8. Retention Priority Score

The Retention Priority Score combines three normalized signals:

- **40% CLV component** — prioritizes economically valuable customers.
- **40% Churn component** — prioritizes customers at greater risk of leaving.
- **20% Segment Ambiguity component** — increases attention for customers whose behavioral segment is uncertain.

The baseline score is:

\[
Retention\ Priority = 0.40(CL V) + 0.40(Churn) + 0.20(Ambiguity)
\]

The score is multiplied by 100 for business interpretability.

These weights represent a transparent project-specific business assumption rather than a learned or universally optimal weighting scheme. Sensitivity analysis is therefore performed later to determine whether reasonable alternative weights substantially change customer prioritization.

In [13]:
# ---------------------------------------------------------
# Baseline retention-priority score
# ---------------------------------------------------------

CLV_WEIGHT = 0.40
CHURN_WEIGHT = 0.40
AMBIGUITY_WEIGHT = 0.20

assert np.isclose(
    CLV_WEIGHT + CHURN_WEIGHT + AMBIGUITY_WEIGHT,
    1.0
)

def calculate_retention_score(df):
    result = df.copy()

    result["retention_priority_score"] = 100 * (
        CLV_WEIGHT * result["clv_component"]
        + CHURN_WEIGHT * result["churn_component"]
        + AMBIGUITY_WEIGHT * result["ambiguity_component"]
    )

    return result


dev_retention = calculate_retention_score(dev_retention)
holdout_retention = calculate_retention_score(holdout_retention)

print("DEVELOPMENT RETENTION SCORE")
ipy_display(
    dev_retention["retention_priority_score"]
    .describe()
    .round(2)
    .to_frame()
)

print("\nHOLDOUT RETENTION SCORE")
ipy_display(
    holdout_retention["retention_priority_score"]
    .describe()
    .round(2)
    .to_frame()
)

assert dev_retention["retention_priority_score"].between(0, 100).all()
assert holdout_retention["retention_priority_score"].between(0, 100).all()

print("\n✓ Retention priority scores successfully calculated.")

DEVELOPMENT RETENTION SCORE


,retention_priority_score
count,40000.00
mean,37.69
std,14.52
min,1.00
25%,26.74
50%,37.53
75%,48.17
max,88.40



HOLDOUT RETENTION SCORE


,retention_priority_score
count,10000.00
mean,37.79
std,14.45
min,1.24
25%,26.89
50%,37.79
75%,48.18
max,88.29



✓ Retention priority scores successfully calculated.


## 9. Weight Sensitivity Analysis

The baseline retention-priority weights (40% CLV, 40% churn risk, and 20% ambiguity) are business-driven assumptions rather than parameters learned from labelled retention outcomes.

To evaluate whether customer prioritization depends excessively on this exact weighting choice, several reasonable alternative weighting schemes are compared.

The analysis examines the rank correlation between the baseline customer ranking and rankings produced by alternative weighting schemes. A high rank correlation indicates that the prioritization is relatively robust to reasonable changes in component weights.

In [14]:
# ---------------------------------------------------------
# Weight sensitivity analysis
# ---------------------------------------------------------

weight_scenarios = {
    "Baseline_40_40_20": (0.40, 0.40, 0.20),
    "Equal_33_33_33": (1/3, 1/3, 1/3),
    "CLV_Focused_50_30_20": (0.50, 0.30, 0.20),
    "Churn_Focused_30_50_20": (0.30, 0.50, 0.20),
    "Ambiguity_Enhanced_35_35_30": (0.35, 0.35, 0.30)
}

sensitivity_scores = pd.DataFrame(
    {"customer_id": dev_retention["customer_id"]}
)

for scenario, (w_clv, w_churn, w_ambiguity) in weight_scenarios.items():

    sensitivity_scores[scenario] = 100 * (
        w_clv * dev_retention["clv_component"]
        + w_churn * dev_retention["churn_component"]
        + w_ambiguity * dev_retention["ambiguity_component"]
    )

baseline_col = "Baseline_40_40_20"

sensitivity_results = []

for scenario in weight_scenarios:

    if scenario == baseline_col:
        continue

    spearman_corr = sensitivity_scores[
        [baseline_col, scenario]
    ].corr(method="spearman").iloc[0, 1]

    sensitivity_results.append({
        "scenario": scenario,
        "spearman_rank_correlation_with_baseline": spearman_corr
    })

sensitivity_results = pd.DataFrame(sensitivity_results)

print("RANK SENSITIVITY RESULTS")
ipy_display(
    sensitivity_results
    .sort_values(
        "spearman_rank_correlation_with_baseline",
        ascending=False
    )
    .round(4)
)

RANK SENSITIVITY RESULTS


,scenario,spearman_rank_correlation_with_baseline
3,Ambiguity_Enhanced_35_35_30,0.9812
1,CLV_Focused_50_30_20,0.9787
0,Equal_33_33_33,0.9656
2,Churn_Focused_30_50_20,0.9653


## 10. Retention Priority Categories

To convert the continuous Retention Priority Score into actionable business groups, thresholds are derived from the development population rather than using arbitrary fixed score values.

Customers are classified as:

- **High Priority:** top 20% of development retention-priority scores.
- **Medium Priority:** customers between the 50th and 80th percentiles.
- **Low Priority:** bottom 50%.

The thresholds are calculated only from the development data and then applied unchanged to the holdout set. This avoids using holdout information to define the prioritization rules.

In [15]:
# ---------------------------------------------------------
# Define retention-priority categories
# using DEVELOPMENT thresholds only
# ---------------------------------------------------------

MEDIUM_THRESHOLD = dev_retention[
    "retention_priority_score"
].quantile(0.50)

HIGH_THRESHOLD = dev_retention[
    "retention_priority_score"
].quantile(0.80)

print(f"Medium threshold (50th percentile): {MEDIUM_THRESHOLD:.2f}")
print(f"High threshold (80th percentile): {HIGH_THRESHOLD:.2f}")


def assign_priority(score):
    if score >= HIGH_THRESHOLD:
        return "High"
    elif score >= MEDIUM_THRESHOLD:
        return "Medium"
    else:
        return "Low"


dev_retention["retention_priority"] = (
    dev_retention["retention_priority_score"]
    .apply(assign_priority)
)

holdout_retention["retention_priority"] = (
    holdout_retention["retention_priority_score"]
    .apply(assign_priority)
)


def priority_summary(df):
    counts = (
        df["retention_priority"]
        .value_counts()
        .reindex(["High", "Medium", "Low"])
    )

    percentages = (
        df["retention_priority"]
        .value_counts(normalize=True)
        .reindex(["High", "Medium", "Low"])
        * 100
    )

    return pd.DataFrame({
        "customers": counts,
        "percentage": percentages.round(2)
    })


print("\nDEVELOPMENT PRIORITY DISTRIBUTION")
ipy_display(priority_summary(dev_retention))

print("\nHOLDOUT PRIORITY DISTRIBUTION")
ipy_display(priority_summary(holdout_retention))

Medium threshold (50th percentile): 37.53
High threshold (80th percentile): 50.69

DEVELOPMENT PRIORITY DISTRIBUTION


,customers,percentage
retention_priority,,
High,8000,20.0
Medium,12000,30.0
Low,20000,50.0



HOLDOUT PRIORITY DISTRIBUTION


,customers,percentage
retention_priority,,
High,2012,20.12
Medium,3056,30.56
Low,4932,49.32


## 11. Validate Retention Priority Groups

The resulting priority categories are examined to confirm that they reflect the intended business logic.

For each priority group, the analysis compares:

- average and median Customer Lifetime Value,
- average churn risk,
- average segment ambiguity,
- average retention-priority score,
- number of customers.

A valid prioritization method should show that High Priority customers have stronger combinations of the signals used to construct the retention score.

In [16]:
# ---------------------------------------------------------
# Validate retention-priority groups
# ---------------------------------------------------------

priority_order = ["High", "Medium", "Low"]

dev_priority_profile = (
    dev_retention
    .groupby("retention_priority")
    .agg(
        customers=("customer_id", "count"),
        mean_clv=("customer_lifetime_value_usd", "mean"),
        median_clv=("customer_lifetime_value_usd", "median"),
        mean_churn_risk=("churn_risk_score", "mean"),
        mean_ambiguity=("segment_ambiguity_score", "mean"),
        mean_priority_score=("retention_priority_score", "mean")
    )
    .reindex(priority_order)
)

holdout_priority_profile = (
    holdout_retention
    .groupby("retention_priority")
    .agg(
        customers=("customer_id", "count"),
        mean_clv=("customer_lifetime_value_usd", "mean"),
        median_clv=("customer_lifetime_value_usd", "median"),
        mean_churn_risk=("churn_risk_score", "mean"),
        mean_ambiguity=("segment_ambiguity_score", "mean"),
        mean_priority_score=("retention_priority_score", "mean")
    )
    .reindex(priority_order)
)

print("DEVELOPMENT PRIORITY PROFILE")
ipy_display(dev_priority_profile.round(2))

print("\nHOLDOUT PRIORITY PROFILE")
ipy_display(holdout_priority_profile.round(2))

DEVELOPMENT PRIORITY PROFILE


,customers,mean_clv,median_clv,mean_churn_risk,mean_ambiguity,mean_priority_score
retention_priority,,,,,,
High,8000,116498.16,111861.72,41.77,0.47,58.35
Medium,12000,76145.53,67310.37,27.21,0.37,43.89
Low,20000,23271.54,17005.34,20.54,0.29,25.71



HOLDOUT PRIORITY PROFILE


,customers,mean_clv,median_clv,mean_churn_risk,mean_ambiguity,mean_priority_score
retention_priority,,,,,,
High,2012,115424.53,109797.24,42.28,0.47,58.22
Medium,3056,74817.86,66633.58,27.39,0.38,43.79
Low,4932,23127.22,16884.90,20.63,0.29,25.74


## 12. Retention Priority Across Customer Segments

Retention priority is analyzed within the four FCM customer segments to determine where high-priority customers are concentrated.

This connects the soft customer segmentation results with the business prioritization layer and helps identify which customer groups may require different retention strategies.

In [17]:
# ---------------------------------------------------------
# Cluster × retention-priority analysis
# ---------------------------------------------------------

cluster_priority_counts = pd.crosstab(
    dev_retention["cluster_name"],
    dev_retention["retention_priority"]
)

cluster_priority_pct = pd.crosstab(
    dev_retention["cluster_name"],
    dev_retention["retention_priority"],
    normalize="index"
) * 100

# Keep category order consistent
for col in ["High", "Medium", "Low"]:
    if col not in cluster_priority_counts.columns:
        cluster_priority_counts[col] = 0
    if col not in cluster_priority_pct.columns:
        cluster_priority_pct[col] = 0

cluster_priority_counts = cluster_priority_counts[
    ["High", "Medium", "Low"]
]

cluster_priority_pct = cluster_priority_pct[
    ["High", "Medium", "Low"]
]

print("CUSTOMER COUNTS BY SEGMENT AND RETENTION PRIORITY")
ipy_display(cluster_priority_counts)

print("\nWITHIN-SEGMENT RETENTION PRIORITY (%)")
ipy_display(cluster_priority_pct.round(2))

CUSTOMER COUNTS BY SEGMENT AND RETENTION PRIORITY


retention_priority,High,Medium,Low
cluster_name,,,
High-Value Active Customers,3841,5001,3629
Inactive / High-Churn-Risk Customers,1650,1865,2216
Low-Engagement Customers,1357,2586,7019
Low-Purchase High-Conversion Customers,1152,2548,7136



WITHIN-SEGMENT RETENTION PRIORITY (%)


retention_priority,High,Medium,Low
cluster_name,,,
High-Value Active Customers,30.80,40.10,29.10
Inactive / High-Churn-Risk Customers,28.79,32.54,38.67
Low-Engagement Customers,12.38,23.59,64.03
Low-Purchase High-Conversion Customers,10.63,23.51,65.85


In [18]:
# ---------------------------------------------------------
# Business profile of the four final FCM segments
# ---------------------------------------------------------

cluster_business_profile = (
    dev_retention
    .groupby("cluster_name")
    .agg(
        customers=("customer_id", "count"),
        mean_clv=("customer_lifetime_value_usd", "mean"),
        mean_churn_risk=("churn_risk_score", "mean"),
        mean_ambiguity=("segment_ambiguity_score", "mean"),
        mean_retention_score=("retention_priority_score", "mean")
    )
    .sort_values("mean_retention_score", ascending=False)
)

cluster_business_profile["percentage"] = (
    cluster_business_profile["customers"]
    / len(dev_retention)
    * 100
)

cluster_business_profile = cluster_business_profile[
    [
        "customers",
        "percentage",
        "mean_clv",
        "mean_churn_risk",
        "mean_ambiguity",
        "mean_retention_score"
    ]
]

ipy_display(cluster_business_profile.round(2))

,customers,percentage,mean_clv,mean_churn_risk,mean_ambiguity,mean_retention_score
cluster_name,,,,,,
High-Value Active Customers,12471,31.18,94795.91,21.55,0.38,44.04
Inactive / High-Churn-Risk Customers,5731,14.33,56615.94,45.79,0.21,42.42
Low-Engagement Customers,10962,27.40,38316.52,25.16,0.37,33.31
Low-Purchase High-Conversion Customers,10836,27.09,35480.90,24.41,0.36,32.31


## 13. Customer-Level Retention Recommendations

The final decision-support output combines:

- FCM customer segment,
- soft membership information,
- Segment Ambiguity Score,
- Customer Lifetime Value,
- Churn Risk,
- Retention Priority Score,
- Retention Priority category.

A simple rule-based recommendation is generated to translate the analytical results into an actionable business message.

These recommendations are decision-support rules and are not predictions learned from labelled campaign-response data.

In [20]:
# ---------------------------------------------------------
# Final customer-level business recommendations
# ---------------------------------------------------------

# Consistent names for the four final FCM clusters
cluster_name_map = {
    1: "Inactive / High-Churn-Risk Customers",
    2: "Low-Purchase High-Conversion Customers",
    3: "High-Value Active Customers",
    4: "Low-Engagement Customers"
}

# Ensure cluster names exist consistently in BOTH datasets
dev_retention["cluster_name"] = (
    dev_retention["assigned_cluster"].map(cluster_name_map)
)

holdout_retention["cluster_name"] = (
    holdout_retention["assigned_cluster"].map(cluster_name_map)
)

# ---------------------------------------------------------
# Define ambiguity threshold independently
# Use the development 75th percentile
# ---------------------------------------------------------

HIGH_AMBIGUITY_THRESHOLD = dev_retention[
    "segment_ambiguity_score"
].quantile(0.75)

print(
    f"High ambiguity threshold "
    f"(development 75th percentile): "
    f"{HIGH_AMBIGUITY_THRESHOLD:.3f}"
)

# Apply the SAME development-derived threshold to holdout
dev_retention["ambiguity_level"] = np.where(
    dev_retention["segment_ambiguity_score"]
    >= HIGH_AMBIGUITY_THRESHOLD,
    "High",
    "Normal"
)

holdout_retention["ambiguity_level"] = np.where(
    holdout_retention["segment_ambiguity_score"]
    >= HIGH_AMBIGUITY_THRESHOLD,
    "High",
    "Normal"
)


# ---------------------------------------------------------
# Generate recommendation
# ---------------------------------------------------------

def generate_recommendation(row):

    priority = row["retention_priority"]
    cluster = row["cluster_name"]
    ambiguity_level = row["ambiguity_level"]

    # High-priority + high-ambiguity customers
    if priority == "High" and ambiguity_level == "High":
        return (
            "High retention priority with uncertain segment membership. "
            "Review mixed customer characteristics and use a personalized "
            "retention strategy."
        )

    # High-priority customers with clearer segment membership
    if priority == "High":

        if cluster == "Inactive / High-Churn-Risk Customers":
            return (
                "High retention priority. Prioritize re-engagement and "
                "churn-prevention actions."
            )

        elif cluster == "High-Value Active Customers":
            return (
                "High retention priority. Protect customer value through "
                "loyalty benefits and personalized retention offers."
            )

        elif cluster == "Low-Engagement Customers":
            return (
                "High retention priority. Use targeted engagement campaigns "
                "to strengthen customer activity."
            )

        elif cluster == "Low-Purchase High-Conversion Customers":
            return (
                "High retention priority. Encourage repeat purchases using "
                "relevant personalized offers."
            )

    # Medium priority
    if priority == "Medium":
        return (
            "Medium retention priority. Monitor the customer and use "
            "targeted engagement where appropriate."
        )

    # Low priority
    return (
        "Low retention priority. Maintain standard engagement and "
        "continue monitoring."
    )


dev_retention["recommendation"] = dev_retention.apply(
    generate_recommendation,
    axis=1
)

holdout_retention["recommendation"] = holdout_retention.apply(
    generate_recommendation,
    axis=1
)


# ---------------------------------------------------------
# Validate
# ---------------------------------------------------------

assert dev_retention["cluster_name"].notna().all()
assert holdout_retention["cluster_name"].notna().all()

assert dev_retention["recommendation"].notna().all()
assert holdout_retention["recommendation"].notna().all()

print("\n✓ Recommendations successfully generated.")

print("\nDevelopment ambiguity levels:")
print(
    dev_retention["ambiguity_level"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nHoldout ambiguity levels:")
print(
    holdout_retention["ambiguity_level"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


# ---------------------------------------------------------
# Display highest-priority customers
# ---------------------------------------------------------

sample_cols = [
    "customer_id",
    "cluster_name",
    "max_membership",
    "segment_ambiguity_score",
    "ambiguity_level",
    "customer_lifetime_value_usd",
    "churn_risk_score",
    "retention_priority_score",
    "retention_priority",
    "recommendation"
]

print("\nTOP 10 RETENTION-PRIORITY CUSTOMERS")

ipy_display(
    dev_retention
    .sort_values(
        "retention_priority_score",
        ascending=False
    )[sample_cols]
    .head(10)
)

High ambiguity threshold (development 75th percentile): 0.531

✓ Recommendations successfully generated.

Development ambiguity levels:
ambiguity_level
Normal    75.0
High      25.0
Name: proportion, dtype: float64

Holdout ambiguity levels:
ambiguity_level
Normal    74.72
High      25.28
Name: proportion, dtype: float64

TOP 10 RETENTION-PRIORITY CUSTOMERS


,customer_id,cluster_name,max_membership,segment_ambiguity_score,ambiguity_level,customer_lifetime_value_usd,churn_risk_score,retention_priority_score,retention_priority,recommendation
38247,CUST-047832,Inactive / High-Churn-Risk Customers,0.469026,0.865800,High,155000.38,84,88.400994,High,High retention priority with uncertain segment...
29799,CUST-037342,High-Value Active Customers,0.352036,0.870758,High,155906.94,82,87.764160,High,High retention priority with uncertain segment...
39955,CUST-049947,High-Value Active Customers,0.678528,0.679205,High,176416.80,87,86.947104,High,High retention priority with uncertain segment...
9801,CUST-012261,High-Value Active Customers,0.368859,0.973944,High,102578.85,87,86.581877,High,High retention priority with uncertain segment...
5936,CUST-007485,High-Value Active Customers,0.458377,0.825957,High,141122.64,83,86.185136,High,High retention priority with uncertain segment...
1359,CUST-001699,High-Value Active Customers,0.446103,0.800360,High,183931.93,78,86.067202,High,High retention priority with uncertain segment...
21209,CUST-026537,Inactive / High-Churn-Risk Customers,0.469024,0.819317,High,157123.12,80,85.996333,High,High retention priority with uncertain segment...
7902,CUST-009931,Inactive / High-Churn-Risk Customers,0.577169,0.726483,High,184262.25,81,85.796651,High,High retention priority with uncertain segment...
37271,CUST-046631,High-Value Active Customers,0.422466,0.928365,High,162938.72,73,85.715302,High,High retention priority with uncertain segment...
6349,CUST-008002,High-Value Active Customers,0.526715,0.746130,High,148231.89,83,85.107593,High,High retention priority with uncertain segment...


## 14. Save Final Retention Prioritization Results

The final customer-level outputs are exported for use in later system development.

The exported results contain:

- FCM cluster memberships,
- assigned customer segment,
- membership confidence,
- Segment Ambiguity Score,
- ambiguity level,
- Customer Lifetime Value,
- churn risk,
- normalized retention components,
- Retention Priority Score,
- Retention Priority category,
- business recommendation.

Separate development and holdout outputs are retained to preserve the evaluation structure used throughout the project.

In [21]:
# ---------------------------------------------------------
# Save final customer-level retention outputs
# ---------------------------------------------------------

final_output_cols = [
    "customer_id",

    "cluster_1_membership",
    "cluster_2_membership",
    "cluster_3_membership",
    "cluster_4_membership",

    "assigned_cluster",
    "cluster_name",
    "max_membership",
    "membership_margin",

    "segment_ambiguity_score",
    "ambiguity_level",

    "customer_lifetime_value_usd",
    "churn_risk_score",

    "clv_component",
    "churn_component",
    "ambiguity_component",

    "retention_priority_score",
    "retention_priority",

    "recommendation"
]

final_dev_output = dev_retention[final_output_cols].copy()
final_holdout_output = holdout_retention[final_output_cols].copy()

dev_output_path = RESULTS_DIR / "final_development_retention_prioritization.csv"
holdout_output_path = RESULTS_DIR / "final_holdout_retention_prioritization.csv"

final_dev_output.to_csv(
    dev_output_path,
    index=False
)

final_holdout_output.to_csv(
    holdout_output_path,
    index=False
)

print("Saved development output:")
print(dev_output_path)

print("\nSaved holdout output:")
print(holdout_output_path)

print("\nDevelopment shape:", final_dev_output.shape)
print("Holdout shape:", final_holdout_output.shape)

print("\nMissing values:")
print(
    "Development:",
    final_dev_output.isna().sum().sum()
)
print(
    "Holdout:",
    final_holdout_output.isna().sum().sum()
)

print("\n✓ Final retention-prioritization outputs saved.")

Saved development output:
d:\FDM\soft-ecommerce-customer-segmentation\results\final_development_retention_prioritization.csv

Saved holdout output:
d:\FDM\soft-ecommerce-customer-segmentation\results\final_holdout_retention_prioritization.csv

Development shape: (40000, 19)
Holdout shape: (10000, 19)

Missing values:
Development: 0
Holdout: 0

✓ Final retention-prioritization outputs saved.


In [22]:
import json

retention_config = {
    "final_clustering_model": {
        "algorithm": "Fuzzy C-Means",
        "n_clusters": 4,
        "fuzziness_m": 1.10,
        "initialization": "K-Means-informed"
    },

    "ambiguity": {
        "method": "normalized_entropy",
        "high_ambiguity_threshold": float(HIGH_AMBIGUITY_THRESHOLD),
        "threshold_source": "development_75th_percentile"
    },

    "retention_score": {
        "clv_weight": CLV_WEIGHT,
        "churn_weight": CHURN_WEIGHT,
        "ambiguity_weight": AMBIGUITY_WEIGHT,
        "score_scale": "0-100"
    },

    "priority_thresholds": {
        "medium_threshold": float(MEDIUM_THRESHOLD),
        "high_threshold": float(HIGH_THRESHOLD),
        "threshold_source": "development_distribution"
    },

    "priority_definition": {
        "high": "score >= development 80th percentile",
        "medium": "development 50th percentile <= score < development 80th percentile",
        "low": "score < development 50th percentile"
    }
}

config_path = RESULTS_DIR / "retention_prioritization_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(
        retention_config,
        f,
        indent=4
    )

print("Configuration saved:")
print(config_path)

print("\nSaved configuration:")
print(
    json.dumps(
        retention_config,
        indent=4
    )
)

Configuration saved:
d:\FDM\soft-ecommerce-customer-segmentation\results\retention_prioritization_config.json

Saved configuration:
{
    "final_clustering_model": {
        "algorithm": "Fuzzy C-Means",
        "n_clusters": 4,
        "fuzziness_m": 1.1,
        "initialization": "K-Means-informed"
    },
    "ambiguity": {
        "method": "normalized_entropy",
        "high_ambiguity_threshold": 0.5311607057521214,
        "threshold_source": "development_75th_percentile"
    },
    "retention_score": {
        "clv_weight": 0.4,
        "churn_weight": 0.4,
        "ambiguity_weight": 0.2,
        "score_scale": "0-100"
    },
    "priority_thresholds": {
        "medium_threshold": 37.525823612710866,
        "high_threshold": 50.69229828700424,
        "threshold_source": "development_distribution"
    },
    "priority_definition": {
        "high": "score >= development 80th percentile",
        "medium": "development 50th percentile <= score < development 80th percentile",
  

## 15. Final Pipeline Validation

A final integrity check is performed before the retention-prioritization results are used by the application.

The validation confirms that:

- all development and holdout customers are retained,
- customer IDs remain unique,
- no development/holdout customer overlap exists,
- no missing values exist in the final outputs,
- membership values remain valid,
- Segment Ambiguity Scores remain within 0–1,
- Retention Priority Scores remain within 0–100,
- priority and ambiguity categories contain only expected values,
- all customers have a business recommendation.

In [23]:
# ---------------------------------------------------------
# Final pipeline integrity validation
# ---------------------------------------------------------

# Correct number of customers
assert len(final_dev_output) == 40000
assert len(final_holdout_output) == 10000

# Unique customer IDs
assert final_dev_output["customer_id"].is_unique
assert final_holdout_output["customer_id"].is_unique

# No development/holdout overlap
final_overlap = (
    set(final_dev_output["customer_id"])
    & set(final_holdout_output["customer_id"])
)
assert len(final_overlap) == 0

# No missing values
assert final_dev_output.isna().sum().sum() == 0
assert final_holdout_output.isna().sum().sum() == 0

# Memberships between 0 and 1
assert (
    final_dev_output[membership_cols]
    .apply(lambda x: x.between(0, 1))
    .all()
    .all()
)

assert (
    final_holdout_output[membership_cols]
    .apply(lambda x: x.between(0, 1))
    .all()
    .all()
)

# Memberships sum to approximately 1
assert np.allclose(
    final_dev_output[membership_cols].sum(axis=1),
    1.0,
    atol=1e-5
)

assert np.allclose(
    final_holdout_output[membership_cols].sum(axis=1),
    1.0,
    atol=1e-5
)

# Ambiguity score range
assert final_dev_output[
    "segment_ambiguity_score"
].between(0, 1).all()

assert final_holdout_output[
    "segment_ambiguity_score"
].between(0, 1).all()

# Retention score range
assert final_dev_output[
    "retention_priority_score"
].between(0, 100).all()

assert final_holdout_output[
    "retention_priority_score"
].between(0, 100).all()

# Valid priority categories
expected_priorities = {"High", "Medium", "Low"}

assert set(
    final_dev_output["retention_priority"].unique()
).issubset(expected_priorities)

assert set(
    final_holdout_output["retention_priority"].unique()
).issubset(expected_priorities)

# Valid ambiguity categories
expected_ambiguity = {"High", "Normal"}

assert set(
    final_dev_output["ambiguity_level"].unique()
).issubset(expected_ambiguity)

assert set(
    final_holdout_output["ambiguity_level"].unique()
).issubset(expected_ambiguity)

# Recommendations exist
assert final_dev_output["recommendation"].str.strip().ne("").all()
assert final_holdout_output["recommendation"].str.strip().ne("").all()

print("✓ Customer counts validated")
print("✓ Customer IDs validated")
print("✓ Development/holdout separation validated")
print("✓ Missing values validated")
print("✓ FCM memberships validated")
print("✓ Segment Ambiguity Scores validated")
print("✓ Retention Priority Scores validated")
print("✓ Priority categories validated")
print("✓ Ambiguity categories validated")
print("✓ Business recommendations validated")

print("\n==========================================")
print("FINAL RETENTION PIPELINE: ALL CHECKS PASSED")
print("==========================================")

✓ Customer counts validated
✓ Customer IDs validated
✓ Development/holdout separation validated
✓ Missing values validated
✓ FCM memberships validated
✓ Segment Ambiguity Scores validated
✓ Retention Priority Scores validated
✓ Priority categories validated
✓ Ambiguity categories validated
✓ Business recommendations validated

FINAL RETENTION PIPELINE: ALL CHECKS PASSED


## 16. Conclusion

The retention-prioritization pipeline was successfully developed using the final selected Fuzzy C-Means clustering solution.

The final workflow:

**Customer Behaviour → FCM Soft Segmentation → Membership Distribution → Segment Ambiguity → CLV + Churn Risk → Retention Priority → Business Recommendation**

The final FCM configuration uses **K = 4**, **m = 1.10**, and **K-Means-informed initialization**.

Normalized membership entropy was used as the **Segment Ambiguity Score**, where higher values represent greater uncertainty in customer segment membership.

Retention priority combines three normalized business signals:

- 40% Customer Lifetime Value
- 40% Churn Risk
- 20% Segment Ambiguity

Sensitivity analysis showed that reasonable alternative weighting schemes produced Spearman rank correlations above 0.96 with the baseline ranking, indicating that the customer prioritization was relatively robust to the tested weighting changes.

Priority thresholds were derived only from the development population and then applied unchanged to holdout customers. The resulting holdout priority distribution and business profiles were similar to development results.

The final outputs provide customer-level soft segment memberships, ambiguity information, retention priority, and interpretable business recommendations for later application integration.